# Gillespie Algorithm: Repressilator and Stochastic Gene Expression

**Authors:** Ania Rotondi and Giulia Marchesani

This notebook compares deterministic and stochastic simulations of a synthetic repressilator and then applies Gillespie SSA to a simple mRNA–protein expression model.

## 1. Setup

The reusable implementations are stored in `src/`. The notebook focuses on simulation, comparison, parameter exploration, and interpretation.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import poisson

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.repressilator import (
    RepressilatorParameters, simulate_deterministic, simulate_gillespie,
    oscillation_metrics, parameter_sweep
)
from src.gene_expression import simulate_ensemble

FIG_DIR = PROJECT_ROOT / "figures"
RESULT_DIR = PROJECT_ROOT / "results"
FIG_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)
np.random.seed(42)

## 2. Deterministic repressilator

The model contains three mRNAs and three proteins. Each protein represses transcription of the next gene in a cyclic loop. ODEs provide smooth trajectories representing average behavior at high molecular copy number.

In [ ]:
params = RepressilatorParameters(
    transcription_rate=50.0, basal_transcription=0.1,
    translation_rate=5.0, mrna_degradation=0.1,
    protein_degradation=0.5, repression_threshold=40.0,
    hill_coefficient=2.0
)
times = np.linspace(0, 200, 2001)
initial_det = [1, 0, 2, 0, 0, 0]
det = simulate_deterministic(times, initial_det, params)

plt.figure(figsize=(10, 5))
for i, label in enumerate(["Protein 1", "Protein 2", "Protein 3"]):
    plt.plot(times, det[:, 3+i], label=label)
plt.xlabel("Time"); plt.ylabel("Protein concentration")
plt.title("Deterministic repressilator"); plt.legend(); plt.show()

oscillation_metrics(times, det[:, 3])

## 3. Deterministic parameter sweep

Transcription rate `alpha` and repression threshold `K` are varied. Period and amplitude summarize how the oscillatory regime changes across parameter space.

In [ ]:
alpha_values = np.linspace(20, 100, 6)
K_values = np.linspace(10, 100, 8)
periods, amplitudes = parameter_sweep(alpha_values, K_values, times, initial_det, params)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, matrix, title in zip(axes, [periods, amplitudes], ["Oscillation period", "Oscillation amplitude"]):
    im = ax.imshow(matrix, origin="lower", aspect="auto", extent=[K_values[0], K_values[-1], alpha_values[0], alpha_values[-1]])
    fig.colorbar(im, ax=ax)
    ax.set(xlabel="K", ylabel="alpha", title=title)
plt.tight_layout(); plt.show()

## 4. Stochastic repressilator

Gillespie SSA represents transcription, degradation, translation, and protein degradation as discrete random events. This reveals intrinsic molecular noise that is absent from the ODE solution.

In [ ]:
initial_stochastic = [1, 1, 1, 0, 0, 0]
stochastic_times, stochastic_states = simulate_gillespie(200, initial_stochastic, params, seed=42)

plt.figure(figsize=(10, 5))
for i, label in enumerate(["Protein 1", "Protein 2", "Protein 3"]):
    plt.step(stochastic_times, stochastic_states[:, 3+i], where="post", label=label)
plt.xlabel("Time"); plt.ylabel("Protein molecule count")
plt.title("Stochastic repressilator"); plt.legend(); plt.show()

oscillation_metrics(stochastic_times, stochastic_states[:, 3])

## 5. Deterministic versus stochastic comparison

The deterministic system shows smooth average dynamics, whereas the stochastic system displays irregular event timing and variable amplitudes. Both models describe the same regulatory logic at different levels of abstraction.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
for i, label in enumerate(["Protein 1", "Protein 2", "Protein 3"]):
    axes[0].plot(times, det[:, 3+i], label=label)
    axes[1].step(stochastic_times, stochastic_states[:, 3+i], where="post", label=label)
axes[0].set_title("Deterministic ODE model"); axes[0].set_ylabel("Concentration")
axes[1].set_title("Gillespie SSA model"); axes[1].set_xlabel("Time"); axes[1].set_ylabel("Molecule count")
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()

## 6. Simple mRNA–protein birth–death model

A second Gillespie model contains transcription, mRNA degradation, translation, and protein degradation. One hundred independent trajectories illustrate variability around the ensemble mean.

In [ ]:
time_points = np.linspace(0, 50, 101)
samples = simulate_ensemble(
    size=100, time_points=time_points, initial_population=(0, 0),
    params=(10.0, 1.0, 10.0, 0.4), seed=42
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for species, label in enumerate(["mRNA", "Protein"]):
    for trajectory in samples[:, :, species]:
        axes[species].plot(time_points, trajectory, linewidth=0.3, alpha=0.15)
    axes[species].plot(time_points, samples[:, :, species].mean(axis=0), linewidth=3, label="Mean")
    axes[species].set(xlabel="Time", ylabel="Copy number", title=label)
    axes[species].legend()
plt.tight_layout(); plt.show()

## 7. Steady-state distributions

For a simple transcription–degradation process, mRNA copy numbers are expected to approach a Poisson distribution under standard assumptions. Protein counts are broader because translation propagates and amplifies mRNA fluctuations.

In [ ]:
steady_mrna = samples[:, -50:, 0].ravel()
steady_protein = samples[:, -50:, 1].ravel()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(steady_mrna, bins=np.arange(steady_mrna.max()+2)-0.5, density=True, alpha=0.7, label="Simulated")
x = np.arange(steady_mrna.max()+1)
axes[0].plot(x, poisson.pmf(x, steady_mrna.mean()), "o-", label="Poisson reference")
axes[0].set(title="mRNA steady-state distribution", xlabel="mRNA count", ylabel="Probability")
axes[0].legend()
axes[1].hist(steady_protein, bins=30, density=True, alpha=0.7)
axes[1].set(title="Protein steady-state distribution", xlabel="Protein count", ylabel="Density")
plt.tight_layout(); plt.show()

## Main conclusions

The ODE model captures the average oscillatory behavior of the repressilator, while Gillespie SSA reveals random timing and amplitude fluctuations. Parameter sweeps show that transcription strength and repression threshold alter oscillation characteristics. In the simple expression model, mRNA is close to a Poisson birth–death process, whereas protein variability is broader because translation amplifies upstream noise.